# Week 8-F: Measured query-conditioned latent adaptation

Compare QCLA-0 (unchanged SFR), QCLA-1 (query-conditioned), and QCLA-2 (query plus **frozen unadapted k4 draft**). Every variant preserves the canonical semantic 4-FULL/6-HEAVY allocation and the original frozen SARA generator/projector. QCLA-1/2 start as exact identity residuals, without output LayerNorm. Real 4,096-dimensional query and draft vectors come from SFR-Embedding-Mistral; missing inputs raise an error.

Train on FIT only: seed 42, two fixed epochs, learning rate 1e-4, accumulation of 16 queries. The first gold reference supplies answer-only NLL targets (up to 127 content tokens plus EOS); generated-answer F1 uses the normal multi-reference scorer. Gold is never an adapter input. No checkpoint/epoch selection uses INTERNAL-VAL. This is previously inspected development confirmation; TEST remains untouched.

**Run All first performs a real two-query smoke experiment. Set `RUN_FULL = True` below to additionally start full training.** Full training is substantial GPU work. W8-A through W8-E are reused, not rerun. Checkpoints resume at epoch boundaries; validation generation resumes per query. Completed results are verified and reused.

In [1]:
import sys, subprocess, json
from pathlib import Path
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'configs/week8/w8_dafir.yaml').is_file())
SCRIPT = ROOT / 'scripts/week8/run_w8f_qcla_experiments.py'
OUTPUT = ROOT / 'results/week8/w8f_qcla'
RUN_FULL = True  # Set True to run full FIT training and validation.
EPOCHS = 2       # Fixed before validation. A new budget requires a new output directory.
def run(*args):
    subprocess.run([sys.executable, str(SCRIPT), *map(str, args)], cwd=ROOT, check=True)
print('Python:', sys.executable)
run('--check', '--epochs', EPOCHS, '--output-dir', OUTPUT)

Python: /home/gunavenkat/Downloads/CS787-RAG-Project/sara_env/bin/python
Verified canonical inputs and split disjointness: {'fit': 1746, 'internal_val': 342}


## Real smoke: conditioning, gradient flow, canonical baseline, generation

Measure SFR vectors for two FIT and two validation queries, train both adapters on FIT, save actual checkpoints, and generate real validation answers for all three variants. QCLA-0 answers must match the saved canonical k4 drafts. Training checks finite, nonzero adapter gradients through frozen SARA. Small-sample smoke numbers validate implementation only; they are not evidence of an improvement.

In [2]:
run('--smoke', '--output-dir', OUTPUT)
smoke = json.loads((OUTPUT / 'smoke/summary.json').read_text())
assert smoke['smoke_only'] is True and smoke['baseline_canonical_match'] is True
display(pd.read_csv(OUTPUT / 'smoke/comparison.csv'))

Verified completed measured W8-F run: /home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8f_qcla/smoke/summary.json


,variant,validation_queries,trainable_params,validation_f1,validation_query_mean_answer_nll,mean_final_prompt_tokens,mean_cumulative_prompt_tokens,mean_adapted_generation_latency_sec,mean_estimated_cumulative_generation_latency_sec,mean_conditioning_encode_seconds,end_to_end_latency_sec
0,qcla_0,2,0,0.153333,1.798693,738.0,738.0,2.762528,2.762528,0.000000,NaN
1,qcla_1,2,52439552,0.154167,1.823067,738.0,738.0,2.604490,2.604490,0.241159,NaN
2,qcla_2,2,62925312,0.153846,1.803692,738.0,1476.0,4.020792,6.963561,0.482318,NaN


## Full fixed-budget experiment

The first run caches the remaining query/draft SFR vectors before loading SARA. Original passage embeddings are reused. QCLA-0 reproduces canonical k4; QCLA-1/2 train on all 1,746 FIT queries and are confirmed on 342 validation queries. All fixed variants are reported; validation never selects an epoch/checkpoint.

The output folder is guarded by input and implementation hashes. Changed code/settings require a new output directory for an explicitly new experiment. Completed runs are inspected rather than silently overwritten.

In [3]:
if RUN_FULL:
    run('--full', '--epochs', EPOCHS, '--output-dir', OUTPUT)
    summary = json.loads((OUTPUT / 'full/summary.json').read_text())
    assert summary['smoke_only'] is False and summary['TEST_used'] is False
    assert summary['no_validation_selection'] is True
    display(pd.read_csv(OUTPUT / 'full/comparison.csv'))
else:
    print('Smoke complete. Set RUN_FULL=True and Run All to start full training.')

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 121.97it/s]
Some weights of XMistralForCausalLM were not initialized from the model checkpoint at mistralai/Mistral-7B-Instruct-v0.2 and are newly initialized: ['projector.projector.0.bias', 'projector.projector.0.weight', 'projector.projector.2.bias', 'projector.projector.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `

qcla_1 epoch 2 measured FIT NLL 1.205395509858796


qcla_2 FIT epoch 1: 100%|██████████| 1746/1746 [2:51:17<00:00,  5.89s/it]  


qcla_2 epoch 1 measured FIT NLL 1.213733507559783


qcla_2 FIT epoch 2: 100%|██████████| 1746/1746 [2:51:35<00:00,  5.90s/it]  


qcla_2 epoch 2 measured FIT NLL 1.205647515628555


qcla_2 measured generation: 100%|██████████| 342/342 [26:25<00:00,  4.63s/it]


variant  validation_queries  trainable_params  validation_f1  validation_query_mean_answer_nll  mean_final_prompt_tokens  mean_cumulative_prompt_tokens  mean_adapted_generation_latency_sec  mean_estimated_cumulative_generation_latency_sec  mean_conditioning_encode_seconds  end_to_end_latency_sec
 qcla_0                 342                 0       0.389061                          1.259202                935.157895                     935.157895                             2.711161                                          2.711161                          0.000000                     NaN
 qcla_1                 342          52439552       0.393799                          1.218129                935.157895                     935.157895                             2.523085                                          2.523085                          0.102399                     NaN
 qcla_2                 342          62925312       0.398994                          1.219554               

,variant,validation_queries,trainable_params,validation_f1,validation_query_mean_answer_nll,mean_final_prompt_tokens,mean_cumulative_prompt_tokens,mean_adapted_generation_latency_sec,mean_estimated_cumulative_generation_latency_sec,mean_conditioning_encode_seconds,end_to_end_latency_sec
0,qcla_0,342,0,0.389061,1.259202,935.157895,935.157895,2.711161,2.711161,0.000000,NaN
1,qcla_1,342,52439552,0.393799,1.218129,935.157895,935.157895,2.523085,2.523085,0.102399,NaN
2,qcla_2,342,62925312,0.398994,1.219554,935.157895,1870.315789,2.555450,5.247229,0.204798,NaN


## Interpretation and costs

Judge quality using generated-answer F1 and paired query/paper comparisons, not NLL alone. Report all fixed variants. The smoke is only a correctness check.

QCLA-2 requires original k4 draft generation before adapted regeneration; cumulative prompt cost counts both passes. Final prompt tokens alone hide that cost. Query/draft encoding time is measured during batched caching and reported separately. Adapted generation time is measured. These components are **not measured end-to-end deployment latency**. Epoch training costs and checkpoint histories are saved. Frozen W8-C/W8-E models and results remain unchanged.

In [4]:
folder = OUTPUT / ('full' if RUN_FULL else 'smoke')
print('Results:', folder / 'comparison.csv')
print('Protocol:', folder / 'protocol_frozen_before_validation.json')
print('Provenance:', folder / 'provenance.json')
print('Adapter checkpoints:', list(folder.glob('qcla_*.pt')))
print('Validation generations:', list(folder.glob('*_internal_val.jsonl')))

Results: /home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8f_qcla/full/comparison.csv
Protocol: /home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8f_qcla/full/protocol_frozen_before_validation.json
Provenance: /home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8f_qcla/full/provenance.json
Adapter checkpoints: [PosixPath('/home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8f_qcla/full/qcla_0.pt'), PosixPath('/home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8f_qcla/full/qcla_1.pt'), PosixPath('/home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8f_qcla/full/qcla_2.pt')]
Validation generations: [PosixPath('/home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8f_qcla/full/qcla_1_internal_val.jsonl'), PosixPath('/home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8f_qcla/full/qcla_0_internal_val.jsonl'), PosixPath('/home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8f_qcla/full/qcla_2_internal_val.jsonl')]
